# Gerardo Solis

# Imports

In [25]:
# imports and libraries used
import requests
import pandas as pd
import glob
import os
from pyprojroot import here
from re import sub

from pandas.errors import EmptyDataError

# NOTE: Data was pulled and ran from a Colab notebook then code transferred here

In [92]:
# Paths
data_path_corn_raw = here("data/corn_county_tidy.csv")
data_path_wheat = here("data/wheat_county_tidy.csv")

# Helper Functions

In [35]:
from re import sub
# CO2e_100yr is a combination of CO₂ + CH₄ + N₂O converted into CO₂-equivalent using 100-year GWP
def subsectors_to_df(subsectors):
    """
    Normalize Climate TRACE 'subsectors' payload into a DataFrame.
    Handles:
      - list[dict]
      - dict of columns -> list values
      - dict with nested lists/dicts (best-effort)
    """
    if isinstance(subsectors, list):
        # Common case: list of row dicts
        return pd.json_normalize([x for x in subsectors if isinstance(x, dict)])

    if isinstance(subsectors, dict):
        # Case A: already "columns -> values"
        try:
            return pd.DataFrame(subsectors)
        except ValueError:
            # Case B: dict of dicts / nested structure
            # Flatten into rows: each item becomes a row with a key
            rows = []
            for k, v in subsectors.items():
                if isinstance(v, list):
                    # maybe a list of row dicts under this key
                    for item in v:
                        if isinstance(item, dict):
                            rows.append({**item, "_key": k})
                        else:
                            rows.append({"value": item, "_key": k})
                elif isinstance(v, dict):
                    rows.append({**v, "_key": k})
                else:
                    rows.append({"value": v, "_key": k})
            return pd.json_normalize(rows)

    raise TypeError(f"Unsupported subsectors type: {type(subsectors)}")

# National level
# National raw data pull_ct_subsectors(year, country="USA")
def pull_ct_subsectors(year: int, month: int | None = None,
                      gas: str = "co2e_100yr", country: str = "USA", limit: int = 25000):
    params = {"year": year, "gas": gas, "limit": limit, "country": country}
    if month is not None:
        params["month"] = month

    data = ct_get("/sources/emissions", params=params)

    df = subsectors_to_df(data.get("subsectors"))

    # Add date if year/month exist
    if "year" in df.columns and "month" in df.columns:
        df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1), errors="coerce")

    # Ensure year exists
    if "year" not in df.columns:
        df["year"] = year

    return df, data

# State monthly totals
def pull_admin_monthly_totals(
    year: int,
    gadmId: str,
    admin_label: str,
    state_name: str = None,
    gases=("co2e_100yr", "n2o"),
    limit: int = 25000,
    verbose: bool = True
):
    if isinstance(gases, str):
        gases = (gases,)

    frames = []
    last_err = None

    for gas in gases:
        try:
            data = ct_get("/sources/emissions", params={"year": year, "gas": gas, "limit": limit, "gadmId": gadmId})
            ts = data.get("totals", {}).get("timeseries", None)

            ts_df = pd.DataFrame(ts)

            ts_df["gadmId"] = gadmId
            ts_df["admin"] = admin_label
            ts_df["date"] = pd.to_datetime(dict(year=ts_df["year"], month=ts_df["month"], day=1), errors="coerce")

            if state_name:
              ts_df["state"] = state_name

            if "gas" not in ts_df.columns:
                ts_df["gas"] = gas

            frames.append(ts_df)

        except Exception as e:
           continue

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

# State yearly subsector
def pull_admin_subsector_summaries(
    year: int,
    gadmId: str,
    admin_label: str,
    gases=("co2e_100yr", "n2o"),
    state_name: str = None,
    limit: int = 25000,
    verbose: bool = True
):
    if isinstance(gases, str):
        gases = (gases,)

    frames = []

    for gas in gases:
        try:
            data = ct_get("/sources/emissions", params={"year": year, "gas": gas, "limit": limit, "gadmId": gadmId})

            # if the key is missing or empty, skip cleanly
            if ("subsectors" not in data) or (not data["subsectors"]):
                continue

            sub_df = subsectors_to_df(data["subsectors"]).drop(columns=["_key"], errors="ignore")

            sub_df["year"] = year
            sub_df["month"] = pd.NA
            sub_df["date"] = pd.NaT
            sub_df["gadmId"] = gadmId
            sub_df["admin"] = admin_label

            if state_name:
              sub_df["state"] = state_name

            if "gas" not in sub_df.columns:
                sub_df["gas"] = gas

            frames.append(sub_df)

        except Exception as e:
          continue

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

# Fetch multiple state or county level emission
def fetch_all_admin_emissions(
    admins_df,
    years,
    gases=("co2e_100yr", "n2o"),
    limit=25000,
    label_col="name",
    state_name=None,
    state_col=None,
):
    """
    One API call per (admin, year, gas). Extracts both:
      - totals['timeseries'] (monthly totals)
      - subsectors (summary breakdown)

    Returns:
      monthly_totals_df
      subsector_summary_df
      failures_df
    """
    monthly_frames = []
    subsector_frames = []
    failures = []

    for _, row in admins_df.iterrows():
        gadmId = row["id"]
        label = row.get(label_col, gadmId)

        # state label for this admin row (optional)
        st = state_name
        if state_col is not None and state_col in admins_df.columns and pd.notna(row.get(state_col)):
            st = row.get(state_col)

        for y in years:
            for gas in gases:
                try:
                    data = ct_get(
                        "/sources/emissions",
                        params={"year": y, "gas": gas, "limit": limit, "gadmId": gadmId},
                    )

                    # # ---- monthly totals (timeseries) ----
                    # We do not need monthly totals since USDA is yearly
                    # ts = data.get("totals", {}).get("timeseries", None)
                    # if ts:
                    #     ts_df = pd.DataFrame(ts)
                    #     if not ts_df.empty:
                    #         ts_df["gadmId"] = gadmId
                    #         ts_df["admin"] = label
                    #         if st is not None:
                    #             ts_df["state"] = st
                    #         # ensure gas exists
                    #         if "gas" not in ts_df.columns:
                    #             ts_df["gas"] = gas
                    #         ts_df["date"] = pd.to_datetime(
                    #             dict(year=ts_df["year"], month=ts_df["month"], day=1),
                    #             errors="coerce",
                    #         )
                    #         monthly_frames.append(ts_df)

                    # ---- subsector summaries ----
                    subs = data.get("subsectors", None)
                    if subs:
                        sub_df = subsectors_to_df(subs).drop(columns=["_key"], errors="ignore")
                        if not sub_df.empty:
                            sub_df["year"] = y
                            sub_df["month"] = pd.NA
                            sub_df["date"] = pd.NaT
                            sub_df["gadmId"] = gadmId
                            sub_df["admin"] = label
                            if st is not None:
                                sub_df["state"] = st
                            if "gas" not in sub_df.columns:
                                sub_df["gas"] = gas
                            subsector_frames.append(sub_df)

                except Exception as e:
                    failures.append(
                        {
                            "label": label,
                            "state": st,
                            "gadmId": gadmId,
                            "year": y,
                            "gas": gas,
                            "error": str(e)[:200],
                        }
                    )
                    print("Failed:", label, st, gadmId, y, gas, "|", str(e)[:120])
    # Monthly totals not needed. Keeping below code incase we need at some point.
    # monthly_totals_df = pd.concat(monthly_frames, ignore_index=True) if monthly_frames else pd.DataFrame()
    subsector_summary_df = pd.concat(subsector_frames, ignore_index=True) if subsector_frames else pd.DataFrame()
    failures_df = pd.DataFrame(failures)

    return subsector_summary_df, failures_df


# Normalizing Names from USDA to Climate Trace

def normalize_name(x):
    x = str(x).upper().strip()

    # Normalize saint variants
    x = sub(r"\bST[.\s]+\b", "SAINT ", x)
    x = sub(r"\bSTE[.\s]+\b", "SAINTE ", x)

    # Remove apostrophes
    x = x.replace("'", "")

    # Remove punctuation
    x = sub(r"[^A-Z0-9 ]", " ", x)

    # Collapse spaces
    x = sub(r"\s+", " ", x).strip()

    return x

# def norm_state_name(x: str) -> str:
#     x = (x or "").strip().lower()
#     x = re.sub(r"[^a-z0-9\s\-]", "", x)
#     x = re.sub(r"\s+", " ", x).strip()
#     return x


def build_target_county_admins_df_simple(state_counties, country_id="USA"):
    """
    Build a DataFrame of county gadmIds matching USDA state/county names.
    Assumes:
      USDA state = "California"
      CT state = "California State"
      USDA county = "Alameda"
      CT county = "Alameda County"
    """

    targets = []
    missing = []

    # Get CT states
    ct_states = pd.DataFrame(ct_get(f"/admins/{country_id}/subdivisions"))

    # Build state lookup: "California" -> gadmId
    ct_states["state_clean"] = (
        ct_states["name"]
        .str.replace(" State", "", regex=False)
        .apply(normalize_name)
        )
    state_lookup = dict(zip(ct_states["state_clean"], ct_states["id"]))

    for state, counties in state_counties.items():
        state_key = normalize_name(state)

        if state_key not in state_lookup:
            missing.append({"state": state, "county": None})
            continue

        state_id = state_lookup[state_key]

        # Get CT counties for that state
        ct_counties = pd.DataFrame(ct_get(f"/admins/{state_id}/subdivisions"))
        # Louisiana has special naming for county
        ct_counties["county_clean"] = (
            ct_counties["name"]
              .str.replace(" County", "", regex=False)
              .str.replace(" Parish", "", regex=False)
              .apply(normalize_name)
        )

        county_lookup = dict(zip(ct_counties["county_clean"], ct_counties["id"]))


        for county in counties:
          county_key = normalize_name(county)

          # Try exact match first
          if county_key in county_lookup:
              cid = county_lookup[county_key]

          # Try compact match (fixes DE KALB vs DEKALB etc.)
          elif county_key.replace(" ", "") in {
              k.replace(" ", ""): v for k, v in county_lookup.items()
          }:
              compact_lookup = {k.replace(" ", ""): v for k, v in county_lookup.items()}
              cid = compact_lookup[county_key.replace(" ", "")]

          # Virginia independent cities (add CITY if missing)
          elif state.upper() == "VIRGINIA" and not county_key.endswith(" CITY"):
              alt = county_key + " CITY"
              if alt in county_lookup:
                  cid = county_lookup[alt]
              else:
                  missing.append({"state": state, "county": county})
                  continue

          # Oglala Lakota fallback
          elif state.upper() == "SOUTH DAKOTA" and county_key == "OGLALA LAKOTA":
              if "SHANNON" in county_lookup:
                  cid = county_lookup["SHANNON"]
              else:
                  missing.append({"state": state, "county": county})
                  continue
          else:
              missing.append({"state": state, "county": county})
              continue

          targets.append({
              "id": cid,
              "name": county,
              "state_name": state
          })


    return pd.DataFrame(targets), pd.DataFrame(missing)

def run_by_state(
    targets_df,
    years,
    gases=("co2e_100yr","n2o"),
    base_dir="/content/drive/MyDrive/data-sci-207/Data",
    limit=25000,
    label_col="name",
    state_col="state_name",
    build_combined=True
):
    os.makedirs(base_dir, exist_ok=True)

    def file_ok(p, min_bytes=10):
        return os.path.exists(p) and os.path.getsize(p) >= min_bytes

    def safe_read_csv(p):
        try:
            if not os.path.exists(p) or os.path.getsize(p) == 0:
                return None
            return pd.read_csv(p)
        except EmptyDataError:
            return None

    y0, y1 = min(years), max(years)
    states = sorted(targets_df[state_col].dropna().unique())
    n_states = len(states)

    for i, st in enumerate(states, start=1):
        st_df = targets_df[targets_df[state_col] == st].copy()

        subsector_path = os.path.join(base_dir, f"ct_match_corn_{st}_subsectors_{y0}_{y1}.csv")
        fail_path      = os.path.join(base_dir, f"ct_match_corn_{st}_failures_{y0}_{y1}.csv")

        # resume-friendly skip (but avoid skipping 0-byte files)
        if file_ok(subsector_path) and os.path.exists(fail_path):
            print(f"[{i}/{n_states}] SKIP (already exists): {st}")
            continue

        subsectors, failures = fetch_all_admin_emissions(
            st_df,
            years=years,
            gases=gases,
            limit=limit,
            label_col=label_col,
            state_col=state_col
        )

        subsectors.to_csv(subsector_path, index=False)
        failures.to_csv(fail_path, index=False)

        print(f"[{i}/{n_states}] DONE: {st} | counties={len(st_df)} | subsector_rows={len(subsectors)} | failures={len(failures)}")

    if not build_combined:
        print("ALL STATES COMPLETED (no combined build).")
        return

    # ---- Build combined files from whatever exists on disk ----
    subsector_files = sorted(glob.glob(os.path.join(base_dir, f"ct_match_corn_*_subsectors_{y0}_{y1}.csv")))
    failure_files   = sorted(glob.glob(os.path.join(base_dir, f"ct_match_corn_*_failures_{y0}_{y1}.csv")))

    combined_subsector_path = os.path.join(base_dir, f"ct_match_corn_ALL_subsectors_{y0}_{y1}.csv")
    combined_failure_path   = os.path.join(base_dir, f"ct_match_corn_ALL_failures_{y0}_{y1}.csv")

    subs_dfs = [safe_read_csv(f) for f in subsector_files]
    subs_dfs = [d for d in subs_dfs if d is not None]
    subsectors_all = pd.concat(subs_dfs, ignore_index=True) if subs_dfs else pd.DataFrame()
    subsectors_all.to_csv(combined_subsector_path, index=False)

    fail_dfs = [safe_read_csv(f) for f in failure_files]
    fail_dfs = [d for d in fail_dfs if d is not None]
    failures_all = pd.concat(fail_dfs, ignore_index=True) if fail_dfs else pd.DataFrame()
    failures_all.to_csv(combined_failure_path, index=False)

    print("ALL STATES COMPLETED.")
    print(
        f"COMBINED SAVED:\n"
        f" - {combined_subsector_path} ({len(subsectors_all)} rows)\n"
        f" - {combined_failure_path} ({len(failures_all)} rows)"
    )

# Climate Trace API

In [37]:
# Constant Variables used 
BASE_URL = "https://api.climatetrace.org/v7"
TIMEOUT = 60
YEARS = [2021, 2022, 2023, 2024]
# climate_trace

In [39]:
# Get request
def ct_get(path, params=None):
    url = f"{BASE_URL}{path}"
    r = requests.get(url, params=params or {}, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()

## Testing API

In [41]:
# small request to see schema
data = ct_get("/sources/emissions", params={"limit": 25})

type(data), (list(data.keys()) if isinstance(data, dict) else "list response")

# Location : Country, Region etc
# Totals: aggregate emissions total
# Sectors: emissions broken down by major economic sectors
  # ex: power, oil-and-gas, agriculture, transport, industry
# SubSectors within :
  # ex: synthetic fertilizer application, rice cultivation, enteric fermentation (cattle), manure management, cropland fires

(dict, ['location', 'totals', 'sectors', 'subsectors'])

## Getting National Data for US

In [43]:
dfs = []

for y in YEARS:
    df_y, _ = pull_ct_subsectors(y)
    print(y, df_y.shape, "| subsectors type:", type(ct_get("/sources/emissions", params={"year": y, "gas": "co2e_100yr", "limit": 25000, "country": "USA"})["subsectors"]))
    dfs.append(df_y)

all_df = pd.concat(dfs, ignore_index=True)
# Save to csv
# all_df.to_csv("ct_USA_subsectors_monthly_2021_2024.csv", index=False)
all_df.shape

2021 (832, 9) | subsectors type: <class 'dict'>
2022 (832, 9) | subsectors type: <class 'dict'>
2023 (832, 9) | subsectors type: <class 'dict'>
2024 (832, 9) | subsectors type: <class 'dict'>


(3328, 9)

## Dropping unwanted columns from normalization (US)

In [45]:
# Keeping columns with only variables of interest
keep = ["year", "month", "date", "gas", "sector", "subsector", "emissionsQuantity"]
if "percentage" in all_df.columns:
    keep.append("percentage")

raw_clean = all_df[keep].copy()

# Saving Raw Monthly Subsectors
# raw_clean.to_csv("/content/drive/MyDrive/colab_exports/data-sci_207_project/ct_USA_subsectors_monthly_2021_2024.csv", index=False)

raw_clean.shape, raw_clean.head()

((3328, 8),
    year  month date         gas              sector               subsector  \
 0   NaN    NaN  NaT  co2e_100yr  mineral-extraction             iron-mining   
 1   NaN    NaN  NaT  co2e_100yr  mineral-extraction          rock-quarrying   
 2   NaN    NaN  NaT  co2e_100yr  mineral-extraction          sand-quarrying   
 3   NaN    NaN  NaT  co2e_100yr       manufacturing  wood-and-wood-products   
 4   NaN    NaN  NaT  co2e_100yr       manufacturing                aluminum   
 
    emissionsQuantity  percentage  
 0       6.197563e+07    0.085471  
 1       1.214345e+06    0.001675  
 2       5.677365e+05    0.000783  
 3       1.574816e+07    0.021718  
 4       4.407581e+08    0.607851  )

In [47]:
# Inspecting years
(raw_clean.groupby("year")["month"].nunique()).sort_index()

year
2021.0    12
2022.0    12
2023.0    12
2024.0    12
Name: month, dtype: int64

In [49]:
# Inspecting Sectors
raw_clean["sector"].value_counts().head(20)

sector
manufacturing             728
agriculture               624
transportation            416
mineral-extraction        312
forestry-and-land-use     312
fossil-fuel-operations    312
waste                     260
buildings                 156
power                     156
fluorinated-gases          52
Name: count, dtype: int64

In [51]:
# Inspecting Subsectors
raw_clean["subsector"].value_counts().head(30)

subsector
iron-mining                                    52
rock-quarrying                                 52
domestic-shipping                              52
domestic-aviation                              52
other-fossil-fuel-operations                   52
non-broadcasting-vessels                       52
heat-plants                                    52
international-shipping                         52
rice-cultivation                               52
copper-mining                                  52
domestic-wastewater-treatment-and-discharge    52
other-energy-use                               52
other-mining-quarrying                         52
other-onsite-fuel-usage                        52
bauxite-mining                                 52
incineration-and-open-burning-of-waste         52
iron-and-steel                                 52
manure-applied-to-soils                        52
other-agricultural-soil-emissions              52
pulp-and-paper                          

## State level Data

In [53]:
# getting gadmId for state
states = ct_get("/admins/USA/subdivisions")
states[:5], len(states)

([{'id': 'USA.1_1',
   'name': 'Alabama State',
   'full_name': 'Alabama State, USA',
   'level': 1,
   'level_0_id': 'USA',
   'level_1_id': 'USA.1_1',
   'level_2_id': ''},
  {'id': 'USA.2_1',
   'name': 'Alaska State',
   'full_name': 'Alaska State, USA',
   'level': 1,
   'level_0_id': 'USA',
   'level_1_id': 'USA.2_1',
   'level_2_id': ''},
  {'id': 'USA.3_1',
   'name': 'Arizona State',
   'full_name': 'Arizona State, USA',
   'level': 1,
   'level_0_id': 'USA',
   'level_1_id': 'USA.3_1',
   'level_2_id': ''},
  {'id': 'USA.4_1',
   'name': 'Arkansas State',
   'full_name': 'Arkansas State, USA',
   'level': 1,
   'level_0_id': 'USA',
   'level_1_id': 'USA.4_1',
   'level_2_id': ''},
  {'id': 'USA.5_1',
   'name': 'California State',
   'full_name': 'California State, USA',
   'level': 1,
   'level_0_id': 'USA',
   'level_1_id': 'USA.5_1',
   'level_2_id': ''}],
 51)

In [55]:
# gadm for CA is ca_id = "USA.5_1"
# Get source emissions for CA
ca_id = "USA.5_1"

data_ca = ct_get(
    "/sources/emissions",
    params={"year": 2024, "gas": "co2e_100yr", "limit": 25000, "gadmId": ca_id}
)

data_ca["location"]

{'name': 'California State', 'gadmId': 'USA.5_1', 'country': 'USA'}

In [57]:
# Subsectors For CA
df_ca = subsectors_to_df(data_ca["subsectors"])

# add identifiers for later merges
df_ca["gadmId"] = ca_id
df_ca["admin_name"] = "California State"

# add date if possible
if "year" in df_ca.columns and "month" in df_ca.columns:
    df_ca["date"] = pd.to_datetime(dict(year=df_ca["year"], month=df_ca["month"], day=1), errors="coerce")

df_ca.head(), df_ca.shape

(               sector               subsector         gas  emissionsQuantity  \
 0               power  electricity-generation  co2e_100yr       3.857334e+07   
 1      transportation  international-aviation  co2e_100yr       1.949275e+07   
 2               power        other-energy-use  co2e_100yr       5.802785e+06   
 3       manufacturing     other-manufacturing  co2e_100yr       8.717002e+06   
 4  mineral-extraction  other-mining-quarrying  co2e_100yr       2.369975e+05   
 
    percentage       _key  year  month   gadmId        admin_name date  
 0    7.371741  summaries   NaN    NaN  USA.5_1  California State  NaT  
 1    3.725254  summaries   NaN    NaN  USA.5_1  California State  NaT  
 2    1.108969  summaries   NaN    NaN  USA.5_1  California State  NaT  
 3    1.665904  summaries   NaN    NaN  USA.5_1  California State  NaT  
 4    0.045293  summaries   NaN    NaN  USA.5_1  California State  NaT  ,
 (832, 11))

## Monthly State Level Data for CA

In [59]:
ca_id = "USA.5_1"
#Monthly total for CA
ca_monthly_total = pd.concat(
    [pull_admin_monthly_totals(y, ca_id, "CA") for y in YEARS],
    ignore_index=True
)

# ca_monthly_total.to_csv("/content/drive/MyDrive/colab_exports/data-sci_207_project/ct_CA_monthly_total_2021_2024.csv", index=False)
ca_monthly_total.shape
ca_monthly_total.head()

,year,month,gas,emissionsQuantity,gadmId,admin,date
0,2021,1,co2e_100yr,5.060798e+07,USA.5_1,CA,2021-01-01
1,2021,2,co2e_100yr,4.747624e+07,USA.5_1,CA,2021-02-01
2,2021,3,co2e_100yr,5.049054e+07,USA.5_1,CA,2021-03-01
3,2021,4,co2e_100yr,4.822078e+07,USA.5_1,CA,2021-04-01
4,2021,5,co2e_100yr,4.821579e+07,USA.5_1,CA,2021-05-01


In [63]:
# Summary of State for the request year
ca_subsector_summary = pd.concat(
    [pull_admin_subsector_summaries(y, ca_id, "CA") for y in YEARS],
    ignore_index=True
)

# ca_subsector_summary.to_csv("/content/drive/MyDrive/colab_exports/data-sci_207_project/ct_CA_subsector_summary_2021_2024.csv", index=False)
ca_subsector_summary.shape
ca_subsector_summary.head()

,sector,subsector,gas,emissionsQuantity,percentage,year,month,date,gadmId,admin
0,forestry-and-land-use,forest-land-fires,co2e_100yr,5.276581e+07,8.926301,2021,NaN,NaT,USA.5_1,CA
1,agriculture,manure-left-on-pasture-cattle,co2e_100yr,1.065526e+06,0.180253,2021,NaN,NaT,USA.5_1,CA
2,transportation,other-transport,co2e_100yr,4.411555e+06,0.746295,2021,NaN,NaT,USA.5_1,CA
3,waste,domestic-wastewater-treatment-and-discharge,co2e_100yr,4.741521e+06,0.802115,2021,NaN,NaT,USA.5_1,CA
4,manufacturing,food-beverage-tobacco,co2e_100yr,1.497951e+07,2.534059,2021,NaN,NaT,USA.5_1,CA


## Applying to Many States

In [65]:
# Get all states id's
states = ct_get("/admins/USA/subdivisions")
states_df = pd.DataFrame(states)

# keep what we need
states_df = states_df[["id", "name", "full_name", "level"]].copy()
states_df.head()

,id,name,full_name,level
0,USA.1_1,Alabama State,"Alabama State, USA",1
1,USA.2_1,Alaska State,"Alaska State, USA",1
2,USA.3_1,Arizona State,"Arizona State, USA",1
3,USA.4_1,Arkansas State,"Arkansas State, USA",1
4,USA.5_1,California State,"California State, USA",1


In [71]:
# Constants needed To test for a few states
# start with a few states to test
# test_states = states_df.head(5).to_dict("records")
all_states = states_df.to_dict("records")

In [75]:
# Running Query for all states
monthly_frames = []
subsector_frames = []

for st in all_states:
    gadmId = st["id"]
    name = st["name"]
    for y in YEARS:
        try:
            monthly_frames.append(pull_admin_monthly_totals(y, gadmId, name))
            subsector_frames.append(pull_admin_subsector_summaries(y, gadmId, name))
        except Exception as e:
            print("Failed:", name, gadmId, y, "|", str(e)[:120])

monthly_totals_all = pd.concat(monthly_frames, ignore_index=True)
subsector_summary_all = pd.concat(subsector_frames, ignore_index=True)

monthly_totals_all.shape, subsector_summary_all.shape

((4896, 7), (339456, 10))

In [ ]:
# Saving Data for All states
# monthly_totals_all.to_csv("ct_states_monthly_totals_2021_2024.csv", index=False)
# subsector_summary_all.to_csv("ct_states_subsector_summaries_2021_2024.csv", index=False)

## Querying By Counties

In [78]:
# California counties
ca_counties = ct_get(f"/admins/{ca_id}/subdivisions")
ca_counties [:5], len(ca_counties )

([{'id': 'USA.5.1_1',
   'name': 'Alameda County',
   'full_name': 'Alameda County, California State, USA',
   'level': 2,
   'level_0_id': 'USA',
   'level_1_id': 'USA.5_1',
   'level_2_id': 'USA.5.1_1'},
  {'id': 'USA.5.2_1',
   'name': 'Alpine County',
   'full_name': 'Alpine County, California State, USA',
   'level': 2,
   'level_0_id': 'USA',
   'level_1_id': 'USA.5_1',
   'level_2_id': 'USA.5.2_1'},
  {'id': 'USA.5.3_1',
   'name': 'Amador County',
   'full_name': 'Amador County, California State, USA',
   'level': 2,
   'level_0_id': 'USA',
   'level_1_id': 'USA.5_1',
   'level_2_id': 'USA.5.3_1'},
  {'id': 'USA.5.4_1',
   'name': 'Butte County',
   'full_name': 'Butte County, California State, USA',
   'level': 2,
   'level_0_id': 'USA',
   'level_1_id': 'USA.5_1',
   'level_2_id': 'USA.5.4_1'},
  {'id': 'USA.5.5_1',
   'name': 'Calaveras County',
   'full_name': 'Calaveras County, California State, USA',
   'level': 2,
   'level_0_id': 'USA',
   'level_1_id': 'USA.5_1',
   'l

In [80]:
# California County Dataframe Ex.
ca_counties_df = pd.DataFrame(ca_counties)
ca_counties_df.head()

,id,name,full_name,level,level_0_id,level_1_id,level_2_id
0,USA.5.1_1,Alameda County,"Alameda County, California State, USA",2,USA,USA.5_1,USA.5.1_1
1,USA.5.2_1,Alpine County,"Alpine County, California State, USA",2,USA,USA.5_1,USA.5.2_1
2,USA.5.3_1,Amador County,"Amador County, California State, USA",2,USA,USA.5_1,USA.5.3_1
3,USA.5.4_1,Butte County,"Butte County, California State, USA",2,USA,USA.5_1,USA.5.4_1
4,USA.5.5_1,Calaveras County,"Calaveras County, California State, USA",2,USA,USA.5_1,USA.5.5_1


In [82]:
# Extracting emissions for a single county
alameda = ca_counties_df[ca_counties_df["name"].str.contains("Alameda", case=False)].iloc[0]
# Getting county id
alameda_id = alameda["id"]
alameda_name = alameda["name"]
# pulling emissions for that county
data_alameda = ct_get(
    "/sources/emissions",
    params={"year": 2024, "gas": "co2e_100yr", "limit": 25000, "gadmId": alameda_id }
)
data_alameda["location"]

{'name': 'Alameda County', 'gadmId': 'USA.5.1_1', 'country': 'USA'}

In [84]:
#Monthly total for alameda county
alameda_monthly_total = pd.concat(
    [pull_admin_monthly_totals(y, alameda_id, alameda_name, state_name="CA") for y in YEARS],
    ignore_index=True
)

alameda_monthly_total.shape
alameda_monthly_total.head()

,year,month,gas,emissionsQuantity,gadmId,admin,date,state
0,2021,1,co2e_100yr,1.207129e+06,USA.5.1_1,Alameda County,2021-01-01,CA
1,2021,2,co2e_100yr,1.115379e+06,USA.5.1_1,Alameda County,2021-02-01,CA
2,2021,3,co2e_100yr,1.190250e+06,USA.5.1_1,Alameda County,2021-03-01,CA
3,2021,4,co2e_100yr,1.165015e+06,USA.5.1_1,Alameda County,2021-04-01,CA
4,2021,5,co2e_100yr,1.168108e+06,USA.5.1_1,Alameda County,2021-05-01,CA


In [ ]:
# For all counties in CA
county_monthly, county_subsectors, county_failures = fetch_all_admin_emissions(ca_counties_df,YEARS)
# Saving Data to CSV
# county_monthly.to_csv("ct_states_monthly_totals_2021_2024.csv", index=False)
# county_subsectors.to_csv("ct_states_subsector_summaries_2021_2024.csv", index=False)
# county_failures.to_csv("failed_to_fetch.csv", index=False)
county_monthly.head()

## Add All counties from corn data

In [86]:
# Inspect states and counties in corn data
df_corn = pd.read_csv(data_path_corn_raw)
df_corn = df_corn[df_corn["county"] != "OTHER COUNTIES"]
df_corn.head()

,Unnamed: 0,year,state,state_abbr,county,commodity,AREA HARVESTED grain,AREA HARVESTED silage,AREA PLANTED,PRODUCTION grain,PRODUCTION silage,YIELD grain,YIELD silage
0,0,2021,ALABAMA,AL,BALDWIN,CORN,7650.0,NaN,7800.0,1180000.0,NaN,154.2,NaN
1,1,2021,ALABAMA,AL,BARBOUR,CORN,2230.0,NaN,2300.0,422000.0,NaN,189.2,NaN
2,2,2021,ALABAMA,AL,BLOUNT,CORN,1350.0,NaN,1400.0,206000.0,NaN,152.6,NaN
3,3,2021,ALABAMA,AL,CALHOUN,CORN,1960.0,NaN,2000.0,314000.0,NaN,160.2,NaN
4,4,2021,ALABAMA,AL,CHEROKEE,CORN,3870.0,NaN,4000.0,673000.0,NaN,173.9,NaN


In [88]:
df_corn["state"].unique()
# We need data for
#  ['ALABAMA', 'ARKANSAS', 'CALIFORNIA', 'COLORADO', 'DELAWARE',
#        'GEORGIA', 'IDAHO', 'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS',
#        'KENTUCKY', 'LOUISIANA', 'MARYLAND', 'MICHIGAN', 'MINNESOTA',
#        'MISSISSIPPI', 'MISSOURI', 'NEBRASKA', 'NEW YORK',
#        'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO', 'OKLAHOMA',
#        'PENNSYLVANIA', 'SOUTH CAROLINA', 'SOUTH DAKOTA', 'TENNESSEE',
#        'TEXAS', 'VIRGINIA', 'WASHINGTON', 'WISCONSIN']
state_counties = (
    df_corn.groupby("state")["county"].unique()
)

# state_counties["ALABAMA"]
len(df_corn["state"].unique()) # 32 states


32

In [94]:
# Inspect wheat data
df_wheat = pd.read_csv(data_path_wheat)
df_wheat.head()

,Unnamed: 0,year,state,state_abbr,county,commodity,AREA HARVESTED,AREA PLANTED,PRODUCTION,YIELD
0,0,2021,IDAHO,ID,BENEWAH,WHEAT,7150.0,7400.0,220000.0,30.8
1,1,2021,IDAHO,ID,BINGHAM,WHEAT,43200.0,45100.0,4575000.0,105.9
2,2,2021,IDAHO,ID,BONNEVILLE,WHEAT,49600.0,52200.0,3184000.0,64.2
3,3,2021,IDAHO,ID,CANYON,WHEAT,4930.0,5100.0,456000.0,92.5
4,4,2021,IDAHO,ID,CARIBOU,WHEAT,27700.0,28700.0,759000.0,27.4


In [96]:
len(df_wheat["state"].unique())

6

## Matching States to the Corn Dataset

In [98]:
targets_df, missing_df = build_target_county_admins_df_simple(state_counties)

targets_df.shape, missing_df.shape
targets_df.head()

,id,name,state_name
0,USA.1.2_1,BALDWIN,ALABAMA
1,USA.1.3_1,BARBOUR,ALABAMA
2,USA.1.5_1,BLOUNT,ALABAMA
3,USA.1.8_1,CALHOUN,ALABAMA
4,USA.1.10_1,CHEROKEE,ALABAMA


In [100]:
# Inspecting missing Counties
missing_df.value_counts()

state     county             
VIRGINIA  CHESAPEAKE CITY        1
          RICHMOND               1
          SUFFOLK CITY           1
          VIRGINIA BEACH CITY    1
Name: count, dtype: int64

In [ ]:
# Query for Each State and combine into a single CSV file
# run_by_state(targets_df, years, build_combined=True)


In [ ]:
# len(data_all)